In [1]:
from utils import *
import math 
from utils import _to_unix, _ms_to_s
import matplotlib
matplotlib.use("Agg")

### 1- Load Data 

In [2]:
data_dir = r'C:\Users\fatum\Documents\EPFL\MA4\MLBD\MLBD_2026\gogymi-dataset-2025-2026-complete\out'
tables, event_tables = load_data(data_dir)

In [3]:
df_math_results = tables['math_results']
df_math_questions = tables['math_questions']
df_text_results = tables['text_results']
df_quiz_results = tables['quiz_results']
df_pageview = tables['pageviews']
df_course_ids = tables['course_ids']
df_students = tables['students']

First, Let's try to see if the early features can help us predict the student's success

### 2- Data Mining and Feature Engineering

In [5]:
# Filtering dfs for early days
early_days = 2

# Convert timestamps to dt 
df_pageview['created_at'] = pd.to_datetime(df_pageview['created_at'], errors = 'coerce')
df_math_results['timestamp'] = pd.to_datetime(df_math_results['timestamp'], errors = 'coerce')
df_text_results['timestamp'] = pd.to_datetime(df_text_results['timestamp'], errors = 'coerce')
df_quiz_results['time'] = pd.to_datetime(df_quiz_results['time'], errors = 'coerce')

In [6]:
student_meta = df_students[['user_id', 'creation_time']].drop_duplicates()
early_ts = {
    row["user_id"]: row["creation_time"] + early_days * 86400
    for _, row in df_students.iterrows()
}

In [7]:
unique_students_ids = set(df_students['user_id'])
unique_students_ids_in_pageview = set(df_pageview['user_id'])

matching_ids = unique_students_ids  & unique_students_ids_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of students: {len(unique_students_ids)}")

Matching: 1691
Original number of students: 1781


In [8]:
df_pageview.dtypes

id                          int64
url                        object
user_id                     int64
created_at    datetime64[ns, UTC]
dtype: object

In [9]:
print("-"*20 + "Before windowing" + "-"*20)
print(f" Pageviews {len(df_pageview)}")
print(f" Math Results {len(df_math_results)}")
print(f" Text Results {len(df_text_results)}")
print(f" Quiz Results {len(df_quiz_results)}")


# Rebuild early_days correctly from UNIX seconds, keeping UTC timezone
df_pageview['early_days'] = pd.to_datetime(df_pageview['user_id'].map(early_ts), unit='s', utc=True)
df_math_results['early_days'] = pd.to_datetime(df_math_results['user_id'].map(early_ts), unit='s')
df_text_results['early_days'] = pd.to_datetime(df_text_results['user_id'].map(early_ts), unit='s')
df_quiz_results['early_days'] = pd.to_datetime(df_quiz_results['user_id'].map(early_ts), unit='s')



# Keep only rows inside the user's first 14 days
early_pageview = df_pageview[df_pageview['created_at'] <= df_pageview['early_days']]
early_math_res = df_math_results[df_math_results['timestamp'] <= df_math_results['early_days']]
early_text_res = df_text_results[df_text_results['timestamp'] <= df_text_results['early_days']]
early_quiz_res = df_quiz_results[df_quiz_results['time'] <= df_quiz_results['early_days']]

print("-"*20 + "After windowing" + "-"*20)

print(f" Pageviews {len(early_pageview)}")
print(f" Math Results {len(early_math_res)}")
print(f" Text Results {len(early_text_res)}")
print(f" Quiz Results {len(early_quiz_res)}")



--------------------Before windowing--------------------
 Pageviews 945171
 Math Results 12060
 Text Results 67351
 Quiz Results 239555
--------------------After windowing--------------------
 Pageviews 23675
 Math Results 10159
 Text Results 40700
 Quiz Results 224239


#### Exercise Diversity Metrics
- Breath Score : ratio of unique questions to total attempts
- Cross Domain Balance : 
--> Use them as student level covariates 

In [10]:
diversity_features = pd.DataFrame({'user_id': df_students['user_id'].unique()})
print(len(diversity_features))

1781


In [11]:
# Unique question ids
math_unique = early_math_res.groupby('user_id')['question_id'].nunique().rename('unique_math_qids')
text_unique = early_text_res.groupby('user_id')['question_id'].nunique().rename('unique_text_qids')
quiz_unique = early_quiz_res.groupby('user_id')['question_id'].nunique().rename('unique_quiz_qids')

diversity_features = diversity_features.merge(math_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(text_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(quiz_unique, on='user_id', how='left').fillna(0)
print(len(diversity_features))

diversity_features

1781


,user_id,unique_math_qids,unique_text_qids,unique_quiz_qids
0,3919,0.0,0.0,0.0
1,359,0.0,0.0,0.0
2,49,0.0,17.0,10.0
3,116,0.0,19.0,172.0
4,118,0.0,18.0,77.0
...,...,...,...,...
1776,6678,0.0,0.0,118.0
1777,6673,0.0,0.0,0.0
1778,6672,0.0,0.0,0.0
1779,6675,0.0,0.0,41.0


#### Shannon Entropy (content focus)
Quantifies how "spread" the student's attention is accross different content categories

In [12]:
import math 
def compute_shannon_entropy(series):
    counts = series.value_counts()
    total = counts.sum()
    if total == 0:
        return 0
    probs = counts / total
    return -sum(p * math.log2(p) for p in probs if p > 0)

In [13]:
early_pageview['url']  = early_pageview['url'].astype('str') 
df_course_ids['url']  = df_course_ids['url'].astype('str') + "/"


In [14]:
tables['course_ids']

,post_id,url,post_type,course_id
0,98,/themen/einfuehrung/,lesson,42
1,100,/themen/schriftliche-addition/,lesson,42
2,102,/themen/schriftliche-subtraktion/,lesson,42
3,104,/themen/schriftliche-multiplikation/,lesson,42
4,106,/themen/schriftliche-division/,lesson,42
...,...,...,...,...
450,24762,/tests/quiz-argumentation/,quiz,3301
451,24787,/tests/quiz-erzaehlung/,quiz,3301
452,24810,/tests/quiz-erzaehlung-2/,quiz,5447
453,24823,/tests/quiz-ueberarbeitung/,quiz,3301


In [15]:
early_pageview = early_pageview.merge(df_course_ids[['url', 'post_type']], on='url', how='left')
print(len(early_pageview))
entropy_features = early_pageview.groupby('user_id')['post_type'].apply(compute_shannon_entropy).rename('content_entropy')

23675


In [16]:
entropy_features

user_id
1029    0.000000
1030    1.156780
1041   -0.000000
1064   -0.000000
1072   -0.000000
          ...   
7159    0.932112
7160    0.337290
7161    0.918296
7164    0.503258
7167    0.881291
Name: content_entropy, Length: 591, dtype: float64

In [17]:
unique_course_url = set(df_course_ids['url'].astype(str))
unique_course_url_in_pageview = set(early_pageview['url'].astype(str))

matching_ids = unique_course_url  & unique_course_url_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of urls: {len(unique_course_url)}")

Matching: 362
Original number of urls: 454


### Session Consistency (using events data)


In [18]:
df_features_events = extract_event_features(event_tables, tables, early_ts)

Lenght before merge : 2002
Lenght after merge : 2005
Lenght before merge : 2005
Lenght after merge : 2005
Lenght before merge : 2005
Lenght after merge : 2005


In [19]:
outcome = build_outcome(tables, early_ts)
df_ml_events   = df_features_events.merge(outcome[["user_id", "label"]],
                                on="user_id", how="inner")
df_ml_events   = df_ml_events.dropna(subset=["label"])
df_ml_events
 
df_final = df_ml_events
dfs = [diversity_features, entropy_features]
for df in dfs: 
    df_final = df_final.merge(df, on='user_id', how='left')

df_final

Index(['Unnamed: 0', 'session_id', 'question_part', 'user_id', 'question_id',
       'points', 'max_points', 'answer', 'timestamp', 'early_days', 'ts',
       'cutoff'],
      dtype='object')
Index(['Unnamed: 0', 'session_id', 'user_id', 'course_id', 'quiz_id',
       'question_id', 'points', 'max_points', 'time', 'hint_count',
       'time_spent', 'correct', 'early_days', 'ts', 'cutoff'],
      dtype='object')
Index(['Unnamed: 0', 'user_id', 'course_id', 'exam_id', 'session_id',
       'question_id', 'points', 'max_points', 'answer', 'feedback',
       'timestamp', 'early_days', 'ts', 'cutoff'],
      dtype='object')

 Outcome median score: 0.629


,index,user_id,clicks__n_total,clicks__avg_scrollY,clicks__session_gap_cv,clicks__scroll_depth_max,heartbeat__n_total,heartbeat__avg_idle_time,heartbeat__scroll_depth_max,heartbeat__avg_scrollY,...,media__total_watch_min,media__n_unique_urls,media__media_type_diversity,media__audio_pct,media__completion_rate,label,unique_math_qids,unique_text_qids,unique_quiz_qids,content_entropy
0,6,113,204.0,0.0,12.061532,0.0,1209.0,20486 days 05:57:30.902083840,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,76.0,71.0,NaN
1,7,127,6256.0,0.0,21.380309,0.0,36045.0,20459 days 10:12:33.666928640,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1,0.0,0.0,201.0,NaN
2,8,141,144.0,0.0,7.875310,0.0,2599.0,20501 days 15:01:54.022169856,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,30.0,0.0,NaN
3,14,525,1053.0,0.0,9.610284,0.0,15692.0,20371 days 00:28:40.368999680,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0,0.0,0.0,67.0,NaN
4,38,563,187.0,0.0,10.790543,0.0,1061.0,20359 days 13:00:38.691584,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1,0.0,0.0,13.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,1999,7159,1007.0,0.0,18.149036,0.0,53795.0,20469 days 05:34:00.037327360,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,36.0,0.932112
1403,2000,7160,2653.0,0.0,20.098609,0.0,14521.0,20421 days 12:38:19.825645312,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1,0.0,0.0,82.0,0.337290
1404,2002,7164,788.0,0.0,20.603465,0.0,2788.0,20409 days 12:40:35.278378496,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1,0.0,0.0,38.0,0.503258
1405,2003,7165,380.0,0.0,6.860790,0.0,1611.0,20419 days 16:03:17.385699840,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,31.0,NaN


In [24]:
df_final.columns

Index(['index', 'user_id', 'clicks__n_total', 'clicks__avg_scrollY',
       'clicks__session_gap_cv', 'clicks__scroll_depth_max',
       'heartbeat__n_total', 'heartbeat__avg_idle_time',
       'heartbeat__scroll_depth_max', 'heartbeat__avg_scrollY',
       'q__n_questions_viewed', 'q__n_unique_urls', 'q__n_unique_courses',
       'q__avg_question_number', 'q__n_active_days', 'q__revisit_rate',
       'media__n_play_events', 'media__n_unique_media',
       'media__total_watch_min', 'media__n_unique_urls',
       'media__media_type_diversity', 'media__audio_pct',
       'media__completion_rate', 'label', 'unique_math_qids',
       'unique_text_qids', 'unique_quiz_qids', 'content_entropy', 'cluster',
       'pca1', 'pca2'],
      dtype='object')

In [ ]:
# Convert timedelta to float64 (seconds)
if "heartbeat__avg_idle_time" in df_ml_events.columns:
    if pd.api.types.is_timedelta64_dtype(df_ml_events["heartbeat__avg_idle_time"]):
        df_ml_events["heartbeat__avg_idle_time"] = (
            df_ml_events["heartbeat__avg_idle_time"].dt.total_seconds().astype("float64")
        )
    else:
        df_ml_events["heartbeat__avg_idle_time"] = pd.to_numeric(
            df_ml_events["heartbeat__avg_idle_time"], errors="coerce"
        ).astype("float64")

df_ml_events["heartbeat__avg_idle_time"].dtype

index                                    int64
user_id                                  int64
clicks__n_total                        float64
clicks__avg_scrollY                    float64
clicks__session_gap_cv                 float64
clicks__scroll_depth_max               float64
heartbeat__n_total                     float64
heartbeat__avg_idle_time       timedelta64[ns]
heartbeat__scroll_depth_max            float64
heartbeat__avg_scrollY                 float64
q__n_questions_viewed                  float64
q__n_unique_urls                       float64
q__n_unique_courses                    float64
q__avg_question_number                 float64
q__n_active_days                       float64
q__revisit_rate                        float64
media__n_play_events                   float64
media__n_unique_media                  float64
media__total_watch_min                 float64
media__n_unique_urls                   float64
media__media_type_diversity            float64
media__audio_

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Prepare your Feature Matrix (df_final)
# Filter to keep only the features for clustering
features_list = [f for f in df_final.columns if f not in ("label", "user_id")]
#features_list = ['unique_math_qids', 'unique_text_qids', 'content_entropy', 'unique_quiz_qids']
X = df_final[features_list].fillna(0) # Handle missing values
X = X.astype('float64')
# 2. Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Perform K-Means Clustering
# We'll choose k=3 to find 'Strugglers', 'Explorers', and 'Specializers'
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_final['cluster'] = clusters

# 4. Dimensionality Reduction for Visualization
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df_final['pca1'] = pca_results[:, 0]
df_final['pca2'] = pca_results[:, 1]

# 5. Create the EDA Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='pca1', y='pca2', 
    hue='cluster', 
    palette='viridis', 
    data=df_final, 
    s=100, alpha=0.7
)

plt.title('Student Behavioral Profiles: Clustering by Early Learning Metrics')
plt.xlabel('Principal Component 1 (General Engagement)')
plt.ylabel('Principal Component 2 (Exercise Diversity)')
plt.legend(title='Student Group')
plt.savefig("PCA.jpg")

TypeError: Cannot cast TimedeltaArray to dtype float64

### PFA for Target Generation

In [ ]:
# Checking Matching indexes
df_math_questions = df_math_questions.rename(columns={'Unnamed: 0' : "question_id"})
df_math_questions['question_id'] = df_math_questions['question_id'].astype(str)

unique_qids = set(df_math_questions['question_id'])
unique_qids_in_results = set(df_math_results['question_id'])

matching_ids = unique_qids  & unique_qids_in_results

print(f"Matching: {len(matching_ids)}")
print(f"Original number of Questions: {len(unique_qids)}")

Matching: 0
Original number of students: 26


In [5]:
import pandas as pd
import statsmodels.formula.api as smf

# Merge results with question tags (KCs)
df_math_questions = df_math_questions.rename(columns={'Unnamed: 0' : "question_id"})
df_math_questions['question_id'] = df_math_questions['question_id'].astype(str)
df = pd.merge(df_math_results, df_math_questions[['question_id', 'tags']], on='question_id')

# Sort by student and time
df = df.sort_values(['user_id', 'timestamp'])

# Create target variable (1 for correct, 0 for incorrect)
df['is_correct'] = (df['points'] / df['max_points'] >= 0.8).astype(int)

# Calculate cumulative successes and failures per student per tag
df['s'] = df.groupby(['user_id', 'tags'])['is_correct'].shift(1).fillna(0).groupby(['user_id', 'tags']).cumsum()
df['f'] = (1 - df.groupby(['user_id', 'tags'])['is_correct'].shift(1).fillna(0)).groupby(['user_id', 'tags']).cumsum()
# Overwrite with simple deterministic dummy values (for testing)


df[['user_id', 'tags', 's', 'f']].head()

KeyError: 'user_id'

In [175]:
formula = "is_correct ~ tags + s:tags + f:tags + s:diversity_score"
model = smf.logit(formula, data=df).fit()

PatsyError: Error evaluating factor: NameError: name 'diversity_score' is not defined
    is_correct ~ tags + s:tags + f:tags + s:diversity_score
                                            ^^^^^^^^^^^^^^^

### 

In [ ]:
math_res = df_math_results['question_id'].astype(str).str.strip().str.lower().str.replace(r'\\.0$', '', regex=True)
math_q_col = df_math_questions['question_id'].astype(str).str.strip().str.lower().str.replace(r'\\.0$', '', regex=True)
math_q_idx = df_math_questions.reset_index()['index'].astype(str).str.strip().str.lower().str.replace(r'\\.0$', '', regex=True)

{
    'math_result_ids_unique': int(math_res.nunique()),
    'math_question_ids_unique_col': int(math_q_col.nunique()),
    'math_question_ids_unique_idx': int(math_q_idx.nunique()),
    'overlap_col': int(len(set(math_res.unique()) & set(math_q_col.unique()))),
    'overlap_idx': int(len(set(math_res.unique()) & set(math_q_idx.unique()))),
    'sample_result_ids': math_res.head(10).tolist(),
    'sample_question_col_ids': math_q_col.head(10).tolist(),
    'sample_question_idx_ids': math_q_idx.head(10).tolist()
}
df_math_pfa = df_math_pfa.sort_values(['user_id', 'tags', 'timestamp'])
df_text_pfa = df_text_pfa.sort_values(['user_id', 'tags', 'timestamp'])

# Previous successes before the current row, per user and tag
df_math_pfa['prior_success'] = df_math_pfa.groupby(['user_id', 'tags'], dropna=False)['math__is_correct'].transform(
    lambda s: s.fillna(0).shift(fill_value=0).cumsum().astype(int)
)
df_text_pfa['prior_success'] = df_text_pfa.groupby(['user_id', 'tags'], dropna=False)['text__is_correct'].transform(
    lambda s: s.fillna(0).shift(fill_value=0).cumsum().astype(int)
)

,Unnamed: 0,session_id,question_part,user_id,question_id,points,max_points,answer,timestamp,course_id,year,tags,math__is_correct,prior_success
1018,1018,67b83c976118412457910511,0,20,66a5eb048948dda8b2fd63d4,0.0,4.0,4.395,1740127383,NaN,NaN,NaN,0,0
1019,1019,67b83c976118412457910511,0,20,66a5eb048948dda8b2fd63da,0.0,4.0,NaN,1740127383,NaN,NaN,NaN,0,0
1020,1020,67b83c976118412457910511,0,20,66a5eb048948dda8b2fd63d8,4.0,4.0,980,1740127383,NaN,NaN,NaN,1,0
1021,1021,67b83c976118412457910511,0,20,66a5eb048948dda8b2fd63d6,0.0,4.0,50m,1740127383,NaN,NaN,NaN,0,1
1022,1022,67b83c976118412457910511,0,20,66a5eb048948dda8b2fd63d5,0.0,4.0,4Fr.,1740127383,NaN,NaN,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11005,11005,69a2aff71366d63fe0425e75,0,7154,6941309132badf41871e2665,0.0,4.0,NaN,1772269559,NaN,NaN,NaN,0,9
11006,11006,69a2aff71366d63fe0425e75,0,7154,6941309132badf41871e2668,0.0,2.0,{},1772269559,NaN,NaN,NaN,0,9
11007,11007,69a2aff71366d63fe0425e75,1,7154,6941309132badf41871e2668,0.0,2.0,{},1772269559,NaN,NaN,NaN,0,9
11008,11008,69a2aff71366d63fe0425e75,0,7154,6941309132badf41871e2666,0.0,2.0,NaN,1772269559,NaN,NaN,NaN,0,9
